# 03 — Statistical Testing

Chi-square tests of independence, Benjamini-Hochberg correction,
and Wilson confidence intervals.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.config import load_config, get_analysis_params
from src.statistics import (
    run_chi_square_tests,
    benjamini_hochberg,
    compute_rule_precision_ci,
)

cfg = load_config()
db_path = cfg['paths']['database']
alpha = get_analysis_params(cfg)['alpha']

In [ ]:
# Run all chi-square tests
chi_results = run_chi_square_tests(db_path, alpha=alpha)

for r in chi_results:
    print(f'\n{r.test_name}')
    print(f'  H0: {r.h0}')
    print(f'  H1: {r.h1}')
    print(f'  chi2 = {r.chi2}, df = {r.dof}')
    print(f'  p-value = {r.p_value:.6e}')
    print(f'  Cramer\'s V = {r.cramers_v}')
    sig = 'REJECT H0 — statistically significant' if r.significant else 'FAIL TO REJECT H0'
    print(f'  Conclusion (alpha={r.alpha}): {sig}')

In [ ]:
# Apply Benjamini-Hochberg FDR correction
p_values = [r.p_value for r in chi_results]
bh_results = benjamini_hochberg(p_values, alpha=alpha)

print('\nBenjamini-Hochberg Correction Results:')
print(f'Number of tests: {len(bh_results)}')
print(f'Alpha: {alpha}')
print()

for r, bh in zip(chi_results, bh_results):
    status = 'SIGNIFICANT' if bh['significant_after_correction'] else 'not significant'
    print(f'  {r.test_name}: p={bh["p_value"]:.4e}, '
          f'BH threshold={bh["bh_threshold"]:.4f}, '
          f'rank={bh["rank"]}, '
          f'{status}')

In [ ]:
# Interpretation guidance
print('\n=== Interpretation Notes ===')
print()
print('1. STATISTICAL vs PRACTICAL SIGNIFICANCE')
print('   With large datasets (millions of alerts), even tiny differences')
print('   become statistically significant. Cramer\'s V measures effect size:')
print('   V < 0.1 = negligible, 0.1-0.3 = small, 0.3-0.5 = medium, >0.5 = large')
print()
print('2. BENJAMINI-HOCHBERG CORRECTION')
print('   When running multiple tests, the chance of at least one false')
print('   positive grows. BH controls the false discovery rate (FDR) —')
print('   the expected proportion of false positives among rejected hypotheses.')
print()
print('3. CAUSALITY')
print('   These tests show association, not causation. A significant result')
print('   means the distributions differ, not that one variable causes the other.')